# **Goodreads Books Dataset - Analysis & Rating Prediction**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostRegressor

In [6]:
books = pd.read_csv('../data/goodreads_books_dataset.csv')

In [7]:
display(books.head())

display(books.info())

display(books.describe())

display(books.isnull().sum())

,rank,percentile_rank,book_id,title,author,rating,rating_category,rating_tier,is_high_rated,title_length,title_complexity,word_count,author_count,author_name_length,has_series_info,series_number,title_type,has_subtitle,has_middle_name,estimated_popularity
0,1,0.0,56859736,Metaphysics of Sound,Nataša Pantović,4.93,Excellent,Tier_10,True,20,Simple,3,1,15,False,NaN,Standard,False,False,High
1,2,0.1,41212190,Learn Spanish with stories and audios as workb...,Anton Hager,4.92,Excellent,Tier_10,True,166,Very Complex,24,1,11,False,NaN,Subtitle,True,False,High
2,3,0.1,38479831,Rivers Never Fill The Sea,Giselle V. Steele,4.88,Excellent,Tier_10,True,25,Moderate,5,1,17,False,NaN,Standard,False,True,High
3,4,0.1,29380718,Secret of the Cassin's Family Curse (Castle of...,Julie-Anne Gamble,4.88,Excellent,Tier_10,True,61,Very Complex,10,1,17,True,1.0,Series,False,False,High
4,5,0.2,35514861,What Healing Should Be: How to relieve pain an...,George Alexandru,4.88,Excellent,Tier_10,True,62,Very Complex,11,1,17,False,NaN,Subtitle,True,False,High


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3045 entries, 0 to 3044
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   rank                  3045 non-null   int64  
 1   percentile_rank       3045 non-null   float64
 2   book_id               3045 non-null   int64  
 3   title                 3045 non-null   object 
 4   author                3045 non-null   object 
 5   rating                3045 non-null   float64
 6   rating_category       3043 non-null   object 
 7   rating_tier           3045 non-null   object 
 8   is_high_rated         3045 non-null   bool   
 9   title_length          3045 non-null   int64  
 10  title_complexity      3045 non-null   object 
 11  word_count            3045 non-null   int64  
 12  author_count          3045 non-null   int64  
 13  author_name_length    3045 non-null   int64  
 14  has_series_info       3045 non-null   bool   
 15  series_number        

None

,rank,percentile_rank,book_id,rating,title_length,word_count,author_count,author_name_length,series_number
count,3045.000000,3045.000000,3.045000e+03,3045.000000,3045.000000,3045.000000,3045.0,3045.000000,1194.000000
mean,1523.000000,50.016420,1.737300e+07,4.057672,32.585550,5.586207,1.0,13.762233,3.484087
std,879.160111,28.872182,3.436127e+07,0.286059,19.980501,3.413018,0.0,3.190402,7.921990
min,1.000000,0.000000,1.100000e+01,0.000000,1.000000,1.000000,1.0,4.000000,0.000000
25%,762.000000,25.000000,9.394800e+04,3.890000,17.000000,3.000000,1.0,12.000000,1.000000
50%,1523.000000,50.000000,1.148385e+06,4.070000,30.000000,5.000000,1.0,13.000000,2.000000
75%,2284.000000,75.000000,2.349650e+07,4.240000,43.000000,7.000000,1.0,16.000000,3.000000
max,3045.000000,100.000000,2.347301e+08,4.930000,166.000000,29.000000,1.0,27.000000,146.000000


rank                       0
percentile_rank            0
book_id                    0
title                      0
author                     0
rating                     0
rating_category            2
rating_tier                0
is_high_rated              0
title_length               0
title_complexity           0
word_count                 0
author_count               0
author_name_length         0
has_series_info            0
series_number           1851
title_type                 0
has_subtitle               0
has_middle_name            0
estimated_popularity       0
dtype: int64

In [8]:
# EDA Visualizations

# 1. Rating Distribution
fig = px.histogram(books, x='rating', nbins=50, 
                   title='Distribution of Book Ratings',
                   labels={'rating': 'Rating', 'count': 'Number of Books'},
                   color_discrete_sequence=['#3498db'])
fig.show()

# 2. Rating by Rating Category
rating_category_stats = books.groupby('rating_category')['rating'].agg(['mean', 'count']).reset_index()
fig = px.bar(rating_category_stats, x='rating_category', y='mean',
             hover_data=['count'], title='Average Rating by Category',
             labels={'mean': 'Average Rating', 'rating_category': 'Rating Category'},
             color='mean', color_continuous_scale='Viridis')
fig.show()

# 3. Rating Tier Distribution
rating_tier_count = books['rating_tier'].value_counts().reset_index()
rating_tier_count.columns = ['rating_tier', 'count']
fig = px.bar(rating_tier_count, x='rating_tier', y='count',
             title='Books Distribution by Rating Tier',
             labels={'rating_tier': 'Rating Tier', 'count': 'Count'},
             color='count', color_continuous_scale='Blues')
fig.show()

# 4. Title Length vs Rating
fig = px.scatter(books, x='title_length', y='rating', 
                 color='is_high_rated', size='word_count',
                 title='Title Length vs Rating',
                 labels={'title_length': 'Title Length (characters)', 'rating': 'Rating'},
                 hover_data=['author', 'title'],
                 color_discrete_map={True: '#2ecc71', False: '#e74c3c'})
fig.show()

# 5. Word Count vs Rating
fig = px.scatter(books, x='word_count', y='rating',
                 color='rating_category', size='author_count',
                 title='Word Count vs Rating',
                 labels={'word_count': 'Word Count', 'rating': 'Rating'})
fig.show()

In [11]:
# More Visualizations

# 6. Author Count Impact
fig = px.box(books, x='author_count', y='rating',
             title='Rating Distribution by Number of Authors',
             labels={'author_count': 'Number of Authors', 'rating': 'Rating'},
             color='author_count',  color_discrete_sequence=['#8e44ad'])
fig.show()

# 7. Has Subtitle Effect
subtitle_impact = books.groupby('has_subtitle')['rating'].agg(['mean', 'count']).reset_index()
subtitle_impact['has_subtitle'] = subtitle_impact['has_subtitle'].map({True: 'With Subtitle', False: 'No Subtitle'})
fig = px.bar(subtitle_impact, x='has_subtitle', y='mean',
             hover_data=['count'], title='Impact of Subtitle on Rating',
             labels={'mean': 'Average Rating', 'has_subtitle': 'Subtitle Status'},
             color='mean', color_continuous_scale='RdYlGn')
fig.show()

# 8. Series vs Non-Series Books
series_impact = books.groupby('has_series_info')['rating'].agg(['mean', 'count']).reset_index()
series_impact['has_series_info'] = series_impact['has_series_info'].map({True: 'Series', False: 'Standalone'})
fig = px.bar(series_impact, x='has_series_info', y='mean',
             hover_data=['count'], title='Series vs Standalone Books Rating',
             labels={'mean': 'Average Rating', 'has_series_info': 'Book Type'},
             color='mean', color_continuous_scale='Blues')
fig.show()

# 9. Title Type Distribution
title_type_rating = books.groupby('title_type')['rating'].agg(['mean', 'count']).reset_index()
fig = px.bar(title_type_rating, x='title_type', y='mean',
             hover_data=['count'], title='Average Rating by Title Type',
             labels={'mean': 'Average Rating', 'title_type': 'Title Type'},
             color='mean', color_continuous_scale='Sunset')
fig.show()

# 10. Title Complexity vs Rating
complexity_rating = books.groupby('title_complexity')['rating'].agg(['mean', 'count']).reset_index()
fig = px.bar(complexity_rating, x='title_complexity', y='mean',
             hover_data=['count'], title='Rating by Title Complexity',
             labels={'mean': 'Average Rating', 'title_complexity': 'Title Complexity'},
             color='mean', color_continuous_scale='Viridis')
fig.show()

In [12]:
# Key Insights Summary
print("=" * 80)
print("GOODREADS BOOKS DATASET - KEY INSIGHTS")
print("=" * 80)

# 1. Rating Statistics
print(f"\n1. RATING STATISTICS:")
print(f"   • Average Rating: {books['rating'].mean():.2f}")
print(f"   • Median Rating: {books['rating'].median():.2f}")
print(f"   • Rating Range: {books['rating'].min():.2f} - {books['rating'].max():.2f}")
print(f"   • Std Dev: {books['rating'].std():.2f}")

# 2. High Rated Books
high_rated_pct = (books['is_high_rated'].sum() / len(books)) * 100
print(f"\n2. HIGH-RATED BOOKS:")
print(f"   • High-Rated Books: {books['is_high_rated'].sum()} ({high_rated_pct:.1f}%)")
print(f"   • Low-Rated Books: {(~books['is_high_rated']).sum()} ({100-high_rated_pct:.1f}%)")

# 3. Category Distribution
print(f"\n3. RATING CATEGORY DISTRIBUTION:")
category_dist = books['rating_category'].value_counts()
for category, count in category_dist.items():
    pct = (count / len(books)) * 100
    print(f"   • {category}: {count} books ({pct:.1f}%)")

# 4. Subtitle Impact
with_subtitle = books[books['has_subtitle']]['rating'].mean()
without_subtitle = books[~books['has_subtitle']]['rating'].mean()
print(f"\n4. SUBTITLE IMPACT ON RATINGS:")
print(f"   • Books with Subtitle: {with_subtitle:.3f} avg rating")
print(f"   • Books without Subtitle: {without_subtitle:.3f} avg rating")
print(f"   • Difference: {abs(with_subtitle - without_subtitle):.3f}")

# 5. Series vs Standalone
series_rating = books[books['has_series_info']]['rating'].mean()
standalone_rating = books[~books['has_series_info']]['rating'].mean()
print(f"\n5. SERIES vs STANDALONE BOOKS:")
print(f"   • Series Books: {series_rating:.3f} avg rating ({books['has_series_info'].sum()} books)")
print(f"   • Standalone Books: {standalone_rating:.3f} avg rating ({(~books['has_series_info']).sum()} books)")

# 6. Author Impact
single_author = books[books['author_count'] == 1]['rating'].mean()
multi_author = books[books['author_count'] > 1]['rating'].mean()
print(f"\n6. AUTHOR COUNT IMPACT:")
print(f"   • Single Author: {single_author:.3f} avg rating")
print(f"   • Multiple Authors: {multi_author:.3f} avg rating")

# 7. Title Length Analysis
avg_title_length = books['title_length'].mean()
print(f"\n7. TITLE CHARACTERISTICS:")
print(f"   • Average Title Length: {avg_title_length:.0f} characters")
print(f"   • Average Word Count (in title): {books['word_count'].mean():.2f} words")

print("\n" + "=" * 80)

GOODREADS BOOKS DATASET - KEY INSIGHTS

1. RATING STATISTICS:
   • Average Rating: 4.06
   • Median Rating: 4.07
   • Rating Range: 0.00 - 4.93
   • Std Dev: 0.29

2. HIGH-RATED BOOKS:
   • High-Rated Books: 1828 (60.0%)
   • Low-Rated Books: 1217 (40.0%)

3. RATING CATEGORY DISTRIBUTION:
   • Excellent: 1828 books (60.0%)
   • Good: 1212 books (39.8%)
   • Fair: 2 books (0.1%)
   • Poor: 1 books (0.0%)

4. SUBTITLE IMPACT ON RATINGS:
   • Books with Subtitle: 4.133 avg rating
   • Books without Subtitle: 4.044 avg rating
   • Difference: 0.088

5. SERIES vs STANDALONE BOOKS:
   • Series Books: 4.086 avg rating (1140 books)
   • Standalone Books: 4.041 avg rating (1905 books)

6. AUTHOR COUNT IMPACT:
   • Single Author: 4.058 avg rating
   • Multiple Authors: nan avg rating

7. TITLE CHARACTERISTICS:
   • Average Title Length: 33 characters
   • Average Word Count (in title): 5.59 words



In [13]:
# Data Preparation for AdaBoost Rating Prediction Model

# Select features for modeling
books_model_data = books.copy()

# Fill missing values
books_model_data['series_number'].fillna(0, inplace=True)
books_model_data['rating_category'].fillna('Unknown', inplace=True)

# Encode categorical variables
le_complexity = LabelEncoder()
le_title_type = LabelEncoder()
le_popularity = LabelEncoder()

books_model_data['title_complexity_encoded'] = le_complexity.fit_transform(books_model_data['title_complexity'])
books_model_data['title_type_encoded'] = le_title_type.fit_transform(books_model_data['title_type'])
books_model_data['estimated_popularity_encoded'] = le_popularity.fit_transform(books_model_data['estimated_popularity'])

# Prepare features for modeling
features_for_model = ['title_length', 'word_count', 'author_count', 'author_name_length', 
                      'title_complexity_encoded', 'title_type_encoded', 'has_subtitle', 
                      'has_series_info', 'has_middle_name', 'series_number', 
                      'percentile_rank', 'estimated_popularity_encoded']

# Convert boolean to int
for col in ['has_subtitle', 'has_series_info', 'has_middle_name']:
    books_model_data[col] = books_model_data[col].astype(int)

X = books_model_data[features_for_model]
y = books_model_data['rating']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {features_for_model}")

Features shape: (3045, 12)
Target shape: (3045,)

Features: ['title_length', 'word_count', 'author_count', 'author_name_length', 'title_complexity_encoded', 'title_type_encoded', 'has_subtitle', 'has_series_info', 'has_middle_name', 'series_number', 'percentile_rank', 'estimated_popularity_encoded']


In [14]:
# Feature Scaling and Model Preparation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=features_for_model)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

# Train AdaBoost Model
print("\nTraining AdaBoost Regressor for Rating Prediction...")

adaboost_model = AdaBoostRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

adaboost_model.fit(X_train, y_train)
print("✓ AdaBoost model trained successfully!")

# Make predictions
y_pred_train = adaboost_model.predict(X_train)
y_pred_test = adaboost_model.predict(X_test)

# Calculate metrics
train_mae = mean_absolute_error(y_train, y_pred_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_r2 = r2_score(y_train, y_pred_train)

test_mae = mean_absolute_error(y_test, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2 = r2_score(y_test, y_pred_test)

print(f"\n{'='*80}")
print(f"ADABOOST RATING PREDICTION MODEL - PERFORMANCE")
print(f"{'='*80}")
print(f"\nTraining Metrics:")
print(f"  • MAE: {train_mae:.4f}")
print(f"  • RMSE: {train_rmse:.4f}")
print(f"  • R² Score: {train_r2:.4f}")

print(f"\nTesting Metrics:")
print(f"  • MAE: {test_mae:.4f}")
print(f"  • RMSE: {test_rmse:.4f}")
print(f"  • R² Score: {test_r2:.4f}")
print(f"\n{'='*80}")

Training set size: 2436
Testing set size: 609

Training AdaBoost Regressor for Rating Prediction...
✓ AdaBoost model trained successfully!

ADABOOST RATING PREDICTION MODEL - PERFORMANCE

Training Metrics:
  • MAE: 0.0378
  • RMSE: 0.0475
  • R² Score: 0.9733

Testing Metrics:
  • MAE: 0.0391
  • RMSE: 0.0553
  • R² Score: 0.9569

✓ AdaBoost model trained successfully!

ADABOOST RATING PREDICTION MODEL - PERFORMANCE

Training Metrics:
  • MAE: 0.0378
  • RMSE: 0.0475
  • R² Score: 0.9733

Testing Metrics:
  • MAE: 0.0391
  • RMSE: 0.0553
  • R² Score: 0.9569



In [18]:
# Model Evaluation Visualizations

# 1. Actual vs Predicted Ratings (Test Set)
predictions_df = pd.DataFrame({
    'Actual Rating': y_test.values,
    'Predicted Rating': y_pred_test,
    'Difference': y_test.values - y_pred_test,
    'Error Percentage': ((y_test.values - y_pred_test) / y_test.values * 100)
})

fig = px.scatter(predictions_df, x='Actual Rating', y='Predicted Rating',
                 hover_data=['Difference', 'Error Percentage'],
                 title='AdaBoost: Actual vs Predicted Book Ratings',
                 labels={'Actual Rating': 'Actual Rating', 'Predicted Rating': 'Predicted Rating'},
                 color='Error Percentage', color_continuous_scale='RdYlGn_r')

# Add perfect prediction line
max_rating = max(predictions_df['Actual Rating'].max(), predictions_df['Predicted Rating'].max())
min_rating = min(predictions_df['Actual Rating'].min(), predictions_df['Predicted Rating'].min())
fig.add_trace(go.Scatter(x=[min_rating, max_rating], y=[min_rating, max_rating],
                         mode='lines', name='Perfect Prediction',
                         line=dict(dash='dash', color='black', width=2)))
fig.show()

# 2. Feature Importance
feature_importance = pd.DataFrame({
    'Feature': features_for_model,
    'Importance': adaboost_model.feature_importances_
}).sort_values('Importance', ascending=False)

fig = px.bar(feature_importance, x='Importance', y='Feature', orientation='h',
             title='Feature Importance in Rating Prediction',
             labels={'Importance': 'Importance Score', 'Feature': 'Feature'},
             color='Importance', color_continuous_scale='Blues')
fig.show()

print(f"\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))


Top 10 Most Important Features:
                     Feature   Importance
             percentile_rank 9.323184e-01
          author_name_length 2.480790e-02
                title_length 2.436886e-02
estimated_popularity_encoded 9.492011e-03
             has_middle_name 3.684245e-03
    title_complexity_encoded 2.747753e-03
                  word_count 2.580864e-03
          title_type_encoded 5.306408e-16
                has_subtitle 3.631372e-17
                author_count 0.000000e+00


In [19]:
# Model Serialization and Prediction Function
import joblib
import os

# Create models directory if it doesn't exist
models_dir = '../models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

# Save the model
model_path = os.path.join(models_dir, 'goodreads_adaboost_rating_model.joblib')
joblib.dump(adaboost_model, model_path)
print(f"✓ AdaBoost model saved to: {model_path}")

# Save the scaler
scaler_path = os.path.join(models_dir, 'goodreads_rating_scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f"✓ Feature scaler saved to: {scaler_path}")

# Save encoders
le_complexity_path = os.path.join(models_dir, 'goodreads_complexity_encoder.joblib')
joblib.dump(le_complexity, le_complexity_path)
print(f"✓ Complexity encoder saved to: {le_complexity_path}")

le_title_type_path = os.path.join(models_dir, 'goodreads_title_type_encoder.joblib')
joblib.dump(le_title_type, le_title_type_path)
print(f"✓ Title type encoder saved to: {le_title_type_path}")

le_popularity_path = os.path.join(models_dir, 'goodreads_popularity_encoder.joblib')
joblib.dump(le_popularity, le_popularity_path)
print(f"✓ Popularity encoder saved to: {le_popularity_path}")

# Save feature columns
feature_cols_path = os.path.join(models_dir, 'goodreads_feature_columns.joblib')
joblib.dump(features_for_model, feature_cols_path)
print(f"✓ Feature columns saved to: {feature_cols_path}")

✓ AdaBoost model saved to: ../models/goodreads_adaboost_rating_model.joblib
✓ Feature scaler saved to: ../models/goodreads_rating_scaler.joblib
✓ Complexity encoder saved to: ../models/goodreads_complexity_encoder.joblib
✓ Title type encoder saved to: ../models/goodreads_title_type_encoder.joblib
✓ Popularity encoder saved to: ../models/goodreads_popularity_encoder.joblib
✓ Feature columns saved to: ../models/goodreads_feature_columns.joblib


In [20]:
# Prediction Function for New Books
def predict_book_rating(title_length, word_count, author_count, author_name_length, 
                        title_complexity, title_type, has_subtitle, has_series_info, 
                        has_middle_name, series_number, percentile_rank, estimated_popularity):
    """
    Predict book rating using the trained AdaBoost model.
    
    Parameters:
    -----------
    title_length : int
        Length of the book title in characters
    word_count : int
        Number of words in the title
    author_count : int
        Number of authors
    author_name_length : int
        Length of author name(s)
    title_complexity : str
        Complexity level ('Simple', 'Moderate', 'Complex', 'Very Complex')
    title_type : str
        Type of title ('Standard', 'Subtitle', 'Series')
    has_subtitle : bool
        Whether the book has a subtitle
    has_series_info : bool
        Whether it's part of a series
    has_middle_name : bool
        Whether author has middle name
    series_number : int
        Series number (0 if not in series)
    percentile_rank : float
        Percentile rank of the book
    estimated_popularity : str
        Estimated popularity ('Low', 'Medium', 'High')
    
    Returns:
    --------
    float : Predicted rating (typically 0-5 scale)
    """
    
    # Create input dataframe
    input_data = pd.DataFrame({
        'title_length': [title_length],
        'word_count': [word_count],
        'author_count': [author_count],
        'author_name_length': [author_name_length],
        'title_complexity_encoded': [le_complexity.transform([title_complexity])[0]],
        'title_type_encoded': [le_title_type.transform([title_type])[0]],
        'has_subtitle': [int(has_subtitle)],
        'has_series_info': [int(has_series_info)],
        'has_middle_name': [int(has_middle_name)],
        'series_number': [series_number],
        'percentile_rank': [percentile_rank],
        'estimated_popularity_encoded': [le_popularity.transform([estimated_popularity])[0]]
    })
    
    # Scale the features
    input_scaled = scaler.transform(input_data)
    
    # Make prediction
    predicted_rating = adaboost_model.predict(input_scaled)[0]
    
    return predicted_rating

# Test the prediction function with sample books
print("\n" + "=" * 80)
print("TESTING RATING PREDICTION FUNCTION - SAMPLE PREDICTIONS")
print("=" * 80)

test_books = [
    {
        "title": "The Great Gatsby",
        "title_length": 15,
        "word_count": 3,
        "author_count": 1,
        "author_name_length": 12,
        "title_complexity": "Simple",
        "title_type": "Standard",
        "has_subtitle": False,
        "has_series_info": False,
        "has_middle_name": True,
        "series_number": 0,
        "percentile_rank": 95.0,
        "estimated_popularity": "High"
    },
    {
        "title": "Harry Potter and the Philosopher's Stone",
        "title_length": 39,
        "word_count": 7,
        "author_count": 1,
        "author_name_length": 12,
        "title_complexity": "Moderate",
        "title_type": "Series",
        "has_subtitle": False,
        "has_series_info": True,
        "has_middle_name": False,
        "series_number": 1,
        "percentile_rank": 99.0,
        "estimated_popularity": "High"
    },
    {
        "title": "A Novel",
        "title_length": 7,
        "word_count": 2,
        "author_count": 1,
        "author_name_length": 8,
        "title_complexity": "Simple",
        "title_type": "Standard",
        "has_subtitle": False,
        "has_series_info": False,
        "has_middle_name": False,
        "series_number": 0,
        "percentile_rank": 25.0,
        "estimated_popularity": "Low"
    }
]

for i, book in enumerate(test_books, 1):
    title = book.pop("title")
    pred_rating = predict_book_rating(**book)
    print(f"\nSample {i}: {title}")
    print(f"  → Predicted Rating: {pred_rating:.2f}/5.0")

print("\n" + "=" * 80)


TESTING RATING PREDICTION FUNCTION - SAMPLE PREDICTIONS

Sample 1: The Great Gatsby
  → Predicted Rating: 3.61/5.0

Sample 2: Harry Potter and the Philosopher's Stone
  → Predicted Rating: 3.56/5.0

Sample 3: A Novel
  → Predicted Rating: 4.25/5.0


Sample 3: A Novel
  → Predicted Rating: 4.25/5.0



/home/tads/Work/TADS_PROJ/30-days-of-datasets/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but AdaBoostRegressor was fitted with feature names

/home/tads/Work/TADS_PROJ/30-days-of-datasets/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but AdaBoostRegressor was fitted with feature names

/home/tads/Work/TADS_PROJ/30-days-of-datasets/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but AdaBoostRegressor was fitted with feature names

